# Docking Preparation

This notebook prepares both the receptor and the ligand library for molecular docking with Vina / QBraz.
It converts files to PDBQT format, adding necessary hydrogens and partial charges.

In [1]:
from openbabel import pybel
from openbabel import openbabel as ob
from pathlib import Path
import sys
import pandas as pd
import os
from rdkit import Chem
from rdkit.Chem import AllChem
import subprocess
import glob
import shutil

## Setup Paths

Define input/output paths and clean up the output directory `data/raw/ligands` to ensure a fresh start.

In [ ]:
receptor_input_path = Path("..") / "data" / "raw" / "3FXI.pdb"
receptor_output_path = Path("..") / "data" / "raw" / "3FXI.pdbqt"

csv_input_path = Path("..") / "data" / "processed" / "best_sepsis_candidates.csv"
pbdqt_folder_path = Path("..") / "data" / "raw" / "ligands"

if pbdqt_folder_path.exists():
    shutil.rmtree(pbdqt_folder_path)

pbdqt_folder_path.mkdir(parents=True, exist_ok=True)

## Receptor Preparation

Prepare the receptor PDB file:
1.  Read the PDB file.
2.  Delete existing hydrogens (to ensure clean state) and re-add them polar hydrogens.
3.  Compute Gasteiger partial charges.
4.  Save as PDBQT.

In [3]:
mol = next(pybel.readfile("pdb", str(receptor_input_path)))
mol.OBMol.DeleteHydrogens()
mol.addh()
charge_model = ob.OBChargeModel.FindType("Gasteiger")
if charge_model:
    success = charge_model.ComputeCharges(mol.OBMol)

conversion_options = {"r": None}
mol.write("pdbqt", str(receptor_output_path), overwrite=True, opt=conversion_options)

print("Receptor Prepared")

## Ligand Library Preparation

Iterate through the top candidates, generating 3D structures and PDBQT files:

### Step 1: RDKit (3D Generation)
-   Convert SMILES to RDKit Molecule.
-   **Add Hydrogens**: Crucial for correct charge calculation and 3D embedding.
-   **Embed**: Generate a random 3D conformer using ETKDGv3.
-   **Optimize**: Refine geometry using MMFF94 force field.

### Step 2: OpenBabel (Charges & Format)
-   Transfer the 3D structure to OpenBabel (via MolBlock).
-   **Compute Charges**: Calculate Gasteiger partial charges (required for PDBQT).
-   **Save**: Write to `.pdbqt` file in the ligands directory.

In [4]:
if input("Are you sure? [y/n] ").lower() == "y":
    df = pd.read_csv(csv_input_path)
    success_count = 0

    for index, row in df.iterrows():
        smiles = row['SMILES']
        ligand_name = f"""ligand_{index:02}_{row["COMB"]}""" 

        # --- PHASE 1: RDKit (3D Generation) ---
        
        mol = Chem.MolFromSmiles(smiles)
        if mol is None: continue

        mol_h = Chem.AddHs(mol)

        params = AllChem.ETKDGv3()
        params.useRandomCoords = True
        embed_res = AllChem.EmbedMolecule(mol_h, params)
        
        if embed_res == -1:
            AllChem.EmbedMolecule(mol_h)
        
        AllChem.MMFFOptimizeMolecule(mol_h)

        mol_block = Chem.MolToMolBlock(mol_h)

        # --- PHASE 2: OpenBabel (Charges & PDBQT) ---
        
        ob_mol = pybel.readstring("mol", mol_block)
        ob_mol.calccharges(model="gasteiger")
        
        output_path = os.path.join(pbdqt_folder_path, f"{ligand_name}.pdbqt")
        ob_mol.write("pdbqt", output_path, overwrite=True)
        
        success_count += 1
        
        if success_count % 10 == 0:
            print(f"Processed: {success_count}")
else:
    print("Aborted by user")